#Cyclistic — Exploración inicial y controles de calidad

##Objetivo

Este notebook contiene los primeros controles para preparar la información respecto al caso de estudio "Cyclistic" del curso "Google Data Analytics Professional Certificate", cuyo objetivo es analizar las diferencias en los patrones de uso de bicicletas compartidas entre usuarios casuales y miembros con suscripción anual.

En este sentido, es necesario explorar previamente los datos e identificar posibles problemas de calidad que puedan afectar los resultados.

##Datos utilizados

Se trabaja con registros de viajes de bicicletas correspondientes al período de septiembre de 2025 a agosto de 2026.
**Aclarcion** Los registros no son de usuarios individuales sino de viajes.
**Entorno de trabajo:** los notebooks fueron desarrollados en Google Colab utilizando archivos almacenados en Google Drive. Las rutas corresponden al entorno original de trabajo y deben adaptarse para ejecutar el análisis en otro entorno.

En esta etapa se utiliza inicialmente el archivo de septiembre de 2025 para explorar la estructura de los datos y definir controles que posteriormente se aplicarán al resto de los meses.

##Controles realizados

Revisión de la estructura y consistencia de los archivos mensuales.

Cálculo y exploración de la duración de los viajes.

Identificación de duraciones negativas, nulas y valores extremos.

Comparación de registros extremos entre usuarios casuales y miembros.

**Propósito:** establecer criterios de validación a los efectos de orientar el proceso posterior de preparación y limpieza.

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
septiembre = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_originales/202509-divvy-tripdata.csv",
    engine="python"
)

## Exploración de la duración de los viajes

Para comenzar, convertimos las columnas de inicio y finalización de los viajes a formato de fecha y hora, y calculamos la duración de cada registro.

Este control nos permite identificar posibles inconsistencias, como viajes con duración negativa o igual a cero, que podrían afectar los cálculos posteriores.

También exploraremos la distribución de las duraciones para identificar valores atípicos y evaluar su posible impacto en el análisis.

In [ ]:
septiembre["started_at"] = pd.to_datetime(septiembre["started_at"])
septiembre["ended_at"] = pd.to_datetime(septiembre["ended_at"])

In [ ]:
((septiembre["ended_at"] - septiembre["started_at"]) <= pd.Timedelta(0)).sum()

np.int64(0)

In [ ]:
septiembre["ride_length"] = septiembre["ended_at"] - septiembre["started_at"]

### Validación de las duraciones

Se realizaron controles para identificar viajes con duraciones negativas o iguales a cero.

Estos registros requieren una revisión especial, ya que pueden responder a inconsistencias en las fechas registradas o a situaciones particulares del funcionamiento del servicio.

En esta etapa buscamos detectar y comprender estas anomalías antes de decidir cómo tratarlas en la preparación definitiva de los datos.

**Criterio:** no eliminar automáticamente los registros anómalos sin investigar previamente sus posibles causas.

In [ ]:
septiembre[["started_at", "ended_at", "ride_length"]].head()

,started_at,ended_at,ride_length
0,2025-09-01 12:05:13.323,2025-09-01 12:25:19.273,0 days 00:20:05.950000
1,2025-09-01 01:41:00.775,2025-09-01 01:48:59.228,0 days 00:07:58.453000
2,2025-09-01 07:16:13.921,2025-09-01 07:39:51.831,0 days 00:23:37.910000
3,2025-09-01 09:31:36.294,2025-09-01 09:49:08.022,0 days 00:17:31.728000
4,2025-09-26 08:51:58.069,2025-09-26 09:07:29.290,0 days 00:15:31.221000


In [ ]:
septiembre["ride_length"].dtype

dtype('<m8[ns]')

In [ ]:
septiembre.sort_values("ride_length", ascending=False)[
    ["started_at", "ended_at", "ride_length"]
].head(20)

,started_at,ended_at,ride_length
99309,2025-09-08 19:07:17.441,2025-09-09 20:07:16.001,1 days 00:59:58.560000
89961,2025-09-04 08:32:19.557,2025-09-05 09:32:17.830,1 days 00:59:58.273000
603979,2025-09-09 13:12:49.317,2025-09-10 14:12:47.325,1 days 00:59:58.008000
101729,2025-09-25 15:57:44.032,2025-09-26 16:57:42.019,1 days 00:59:57.987000
617125,2025-09-24 23:50:09.776,2025-09-26 00:50:07.705,1 days 00:59:57.929000
607831,2025-09-21 10:09:19.087,2025-09-22 11:09:17.000,1 days 00:59:57.913000
613373,2025-09-08 05:49:40.003,2025-09-09 06:49:37.659,1 days 00:59:57.656000
68246,2025-09-18 15:54:41.061,2025-09-19 16:54:38.688,1 days 00:59:57.627000
615500,2025-09-09 06:55:04.685,2025-09-10 07:55:02.210,1 days 00:59:57.525000
90287,2025-09-06 19:29:12.883,2025-09-07 20:29:10.402,1 days 00:59:57.519000


In [ ]:
septiembre["ride_length_min"] = septiembre["ride_length"].dt.total_seconds() / 60

## Exploración de valores extremos

Una vez revisadas las duraciones negativas y nulas, analizamos la distribución de los tiempos de viaje para identificar posibles valores extremos.

Para ello, utilizamos estadísticas descriptivas y percentiles, buscando comprender la dispersión de las duraciones y detectar registros que podrían influir considerablemente en los promedios.

A partir de esta exploración, adoptamos como criterio identificar los viajes con duración inferior a 1 minuto o superior a 120 minutos.

Estos límites son una decisión metodológica del análisis, no una regla que permita afirmar que todos esos registros son incorrectos.

Por ese motivo, los viajes extremos se identifican mediante una variable, en lugar de eliminarlos automáticamente. Esto permite comparar posteriormente los resultados con y sin esos registros.

In [ ]:
septiembre["ride_length_min"].describe()

,ride_length_min
count,714759.000000
mean,16.056040
std,50.552594
min,0.000767
25%,5.720425
50%,9.890967
75%,17.207975
max,1499.976000


In [ ]:
septiembre["ride_length_min"].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

,ride_length_min
0.900,28.842287
0.950,40.088135
0.990,87.162531
0.995,122.441540
0.999,1038.268448


In [ ]:
print("Más de 1 hora:", (septiembre["ride_length_min"] > 60).sum())
print("Más de 2 horas:", (septiembre["ride_length_min"] > 120).sum())
print("Más de 6 horas:", (septiembre["ride_length_min"] > 360).sum())
print("Más de 12 horas:", (septiembre["ride_length_min"] > 720).sum())
print("Más de 24 horas:", (septiembre["ride_length_min"] > 1440).sum())

Más de 1 hora: 15281
Más de 2 horas: 3712
Más de 6 horas: 1006
Más de 12 horas: 820
Más de 24 horas: 618


In [ ]:
septiembre["ride_length_sec"] = septiembre["ride_length"].dt.total_seconds()
print("Menos de 1 minuto:", (septiembre["ride_length_sec"] < 60).sum())
print("Menos de 30 segundos:", (septiembre["ride_length_sec"] < 30).sum())
print("Menos de 10 segundos:", (septiembre["ride_length_sec"] < 10).sum())

Menos de 1 minuto: 18293
Menos de 30 segundos: 13215
Menos de 10 segundos: 3878


In [ ]:
import glob

## Validación de los archivos mensuales

Luego de realizar los controles iniciales sobre septiembre de 2025, revisamos los 12 archivos correspondientes al período de estudio.

El objetivo es comprobar que todos los archivos tengan una estructura compatible para poder consolidarlos posteriormente en un único conjunto de datos.

Para ello, verificamos:

- La cantidad de archivos disponibles.
- La cantidad de filas y columnas de cada archivo.
- La consistencia de los nombres de las columnas entre los distintos meses.

Esta revisión permite identificar posibles diferencias estructurales que podrían generar problemas durante la consolidación.

La preparación y limpieza definitiva de los 12 meses se realiza en el siguiente notebook del proyecto.

In [ ]:
ruta = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_originales/*.csv"

archivos = glob.glob(ruta)

len(archivos)

12

In [ ]:
import os

for archivo in archivos:
    df = pd.read_csv(archivo, engine="python")
    print(os.path.basename(archivo), df.shape)

202509-divvy-tripdata.csv (714759, 13)
202510-divvy-tripdata.csv (646039, 13)
202511-divvy-tripdata.csv (356628, 13)
202512-divvy-tripdata.csv (140534, 13)
202601-divvy-tripdata.csv (137787, 13)
202602-divvy-tripdata.csv (201450, 13)
202603-divvy-tripdata.csv (317037, 13)
202604-divvy-tripdata.csv (448252, 13)
202605-divvy-tripdata.csv (653704, 13)
202606-divvy-tripdata.csv (762550, 13)
202607-divvy-tripdata.csv (869051, 13)
202608-divvy-tripdata.csv (868191, 13)


In [ ]:
for archivo in archivos:
    df = pd.read_csv(archivo, engine="python")
    print(os.path.basename(archivo))
    print(df.columns.tolist())
    print()

202509-divvy-tripdata.csv
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

202510-divvy-tripdata.csv
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

202511-divvy-tripdata.csv
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

202512-divvy-tripdata.csv
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

202601-divvy-tripdata.csv
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_stat

In [ ]:
columnas_referencia = pd.read_csv(archivos[0], engine="python", nrows=0).columns.tolist()

todas_iguales = True

for archivo in archivos:
    columnas = pd.read_csv(archivo, engine="python", nrows=0).columns.tolist()

    if columnas != columnas_referencia:
        todas_iguales = False
        print("Estructura diferente:", os.path.basename(archivo))

print("¿Todos los archivos tienen la misma estructura?", todas_iguales)

¿Todos los archivos tienen la misma estructura? True


## Comparación de valores extremos por tipo de usuario

Una vez definido el criterio para identificar viajes extremos, analizamos su distribución según el tipo de usuario: casual o miembro.

El objetivo es determinar si los valores extremos afectan de manera similar a ambos grupos o si tienen una mayor presencia en alguno de ellos.

Para realizar una comparación adecuada, calculamos la proporción de viajes extremos dentro de cada categoría de usuario, en lugar de comparar únicamente las cantidades absolutas.

Este control es relevante porque una distribución desigual de los valores extremos podría influir en las diferencias observadas entre las duraciones promedio de ambos grupos.

Los resultados de esta exploración servirán como referencia para el análisis posterior, donde compararemos las duraciones incluyendo y excluyendo estos registros.

In [ ]:
for grupo in ["member", "casual"]:
    datos = septiembre[septiembre["member_casual"] == grupo]

    print("\n", grupo.upper())
    print("Total de viajes:", len(datos))

    print("Menos de 1 minuto:", (datos["ride_length_min"] < 1).sum())
    print("Menos de 30 segundos:", (datos["ride_length_sec"] < 30).sum())
    print("Menos de 10 segundos:", (datos["ride_length_sec"] < 10).sum())

    print("Más de 1 hora:", (datos["ride_length_min"] > 60).sum())
    print("Más de 2 horas:", (datos["ride_length_min"] > 120).sum())
    print("Más de 6 horas:", (datos["ride_length_min"] > 360).sum())
    print("Más de 12 horas:", (datos["ride_length_min"] > 720).sum())
    print("Más de 24 horas:", (datos["ride_length_min"] > 1440).sum())


 MEMBER
Total de viajes: 449298
Menos de 1 minuto: 8248
Menos de 30 segundos: 5936
Menos de 10 segundos: 2386
Más de 1 hora: 2859
Más de 2 horas: 737
Más de 6 horas: 199
Más de 12 horas: 128
Más de 24 horas: 90

 CASUAL
Total de viajes: 265461
Menos de 1 minuto: 10045
Menos de 30 segundos: 7279
Menos de 10 segundos: 1492
Más de 1 hora: 12422
Más de 2 horas: 2975
Más de 6 horas: 807
Más de 12 horas: 692
Más de 24 horas: 528


In [ ]:
for grupo in ["member", "casual"]:
    datos = septiembre[septiembre["member_casual"] == grupo]

    print("\n", grupo.upper())
    print("Total de viajes:", len(datos))

    print("Menos de 1 minuto:", (datos["ride_length_min"] < 1).sum() / len(datos) * 100, "%")
    print("Menos de 30 segundos:", (datos["ride_length_sec"] < 30).sum() / len(datos) * 100, "%")
    print("Menos de 10 segundos:", (datos["ride_length_sec"] < 10).sum() / len(datos) * 100, "%")

    print("Más de 1 hora:", (datos["ride_length_min"] > 60).sum() / len(datos) * 100, "%")
    print("Más de 2 horas:", (datos["ride_length_min"] > 120).sum() / len(datos) * 100, "%")
    print("Más de 6 horas:", (datos["ride_length_min"] > 360).sum() / len(datos) * 100, "%")
    print("Más de 12 horas:", (datos["ride_length_min"] > 720).sum() / len(datos) * 100, "%")
    print("Más de 24 horas:", (datos["ride_length_min"] > 1440).sum() / len(datos) * 100, "%")


 MEMBER
Total de viajes: 449298
Menos de 1 minuto: 1.8357526630432366 %
Menos de 30 segundos: 1.3211721396489635 %
Menos de 10 segundos: 0.5310506612537781 %
Más de 1 hora: 0.6363260018962915 %
Más de 2 horas: 0.16403367030345115 %
Más de 6 horas: 0.044291316676237154 %
Más de 12 horas: 0.028488887108333444 %
Más de 24 horas: 0.020031248748046952 %

 CASUAL
Total de viajes: 265461
Menos de 1 minuto: 3.7839833346517944 %
Menos de 30 segundos: 2.742022368634187 %
Menos de 10 segundos: 0.5620411284520136 %
Más de 1 hora: 4.679406767849137 %
Más de 2 horas: 1.1206919283811936 %
Más de 6 horas: 0.30399945754743635 %
Más de 12 horas: 0.26067859308900365 %
Más de 24 horas: 0.1988992733395866 %


In [ ]:
septiembre["extremo"] = (
    (septiembre["ride_length_min"] < 1) |
    (septiembre["ride_length_min"] > 120)
).astype(int)

septiembre["extremo"].value_counts()

,count
extremo,
0,692754
1,22005


In [ ]:
septiembre.groupby("member_casual")["extremo"].mean() * 100

,extremo
member_casual,
casual,4.904675
member,1.999786
